In [1]:

%pip install -q -U \
    transformers \
    accelerate \
    bitsandbytes \
    langchain \
    langchain-core \
    langchain-huggingface \
    langchain-community \
    sentence-transformers \
    faiss-cpu \
    pandas \
    numpy \
    scipy \
    scikit-learn \
    openpyxl \
    pydantic \
    plotly \
    streamlit \
    pyngrok


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 111.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.4/165.4 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.0/573.0 kB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 102.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 102.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 109.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 101.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:

%pip install --no-cache-dir --force-reinstall "numpy==2.2.6"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 229.2 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.5.3
    Uninstalling numpy-2.5.3:
      Successfully uninstalled numpy-2.5.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.6 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.6 which is incompatible.
cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.6 which is incompatible.


In [1]:

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.float16
)

model.eval()

def ask_qwen(prompt, max_new_tokens=250):
    messages = [
        {
            "role": "system",
            "content": (
                "You are Intellianalyzer, a concise and accurate "
                "data analysis assistant. Never invent statistics."
            )
        },
        {"role": "user", "content": prompt}
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

print(ask_qwen(
    "Give me three useful questions to investigate in a customer churn dataset."
))


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Certainly. Here are three useful questions to investigate in a customer churn dataset:

1. **What customer demographics or behavioral patterns are most strongly associated with churn?**  
   *Purpose:* Identify key segments (e.g., age, location, usage frequency, plan type) that correlate with higher churn risk.

2. **How does customer engagement (e.g., login frequency, feature usage, support tickets) change before churn occurs?**  
   *Purpose:* Detect early warning signals by analyzing behavioral trends leading up to churn.

3. **What is the impact of recent service interactions (e.g., complaints, support tickets, billing issues) on churn probability?**  
   *Purpose:* Understand whether specific service touchpoints or customer support events predict churn.

These questions help uncover root causes and enable targeted retention strategies.


In [2]:

import pandas as pd
import numpy as np
import json
from pathlib import Path

def load_dataset(file):
    name = file.name.lower() if hasattr(file, "name") else str(file).lower()

    if name.endswith(".csv"):
        return pd.read_csv(file)

    if name.endswith((".xlsx", ".xls")):
        return pd.read_excel(file)

    raise ValueError("Please upload a CSV or Excel file.")


def profile_dataset(df):
    if df.empty:
        raise ValueError("The uploaded dataset contains no rows.")

    if len(df.columns) == 0:
        raise ValueError("The dataset contains no columns.")

    # Avoid ambiguous duplicate column names in downstream analysis.
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]

    if df.columns.duplicated().any():
        raise ValueError(
            "Duplicate column names found. Rename them before analysis."
        )

    numeric = df.select_dtypes(include=np.number)
    categorical = df.select_dtypes(
        include=["object", "category", "bool", "string"]
    )

    report = {
        "dataset": {
            "rows": int(len(df)),
            "columns": int(len(df.columns)),
            "duplicate_rows": int(df.duplicated().sum()),
            "memory_mb": round(
                df.memory_usage(deep=True).sum() / (1024**2), 2
            ),
        },
        "columns": {},
        "correlations": {},
        "warnings": []
    }

    for col in df.columns:
        series = df[col]
        info = {
            "dtype": str(series.dtype),
            "missing_count": int(series.isna().sum()),
            "missing_percent": round(
                float(series.isna().mean() * 100), 2
            ),
            "unique_values": int(series.nunique(dropna=True))
        }

        if col in numeric.columns:
            values = series.replace([np.inf, -np.inf], np.nan).dropna()

            info["statistics"] = {
                "mean": float(values.mean()) if len(values) else None,
                "median": float(values.median()) if len(values) else None,
                "std": float(values.std()) if len(values) > 1 else None,
                "min": float(values.min()) if len(values) else None,
                "max": float(values.max()) if len(values) else None,
                "q25": float(values.quantile(0.25)) if len(values) else None,
                "q75": float(values.quantile(0.75)) if len(values) else None
            }

            if len(values) and values.nunique() <= 1:
                report["warnings"].append(
                    f"{col} is constant or has only one valid value."
                )

        else:
            # Limit category summaries to the 10 most frequent values.
            counts = series.value_counts(dropna=True).head(10)
            info["top_values"] = {
                str(k): int(v) for k, v in counts.items()
            }

        report["columns"][str(col)] = info

    if len(numeric.columns) >= 2:
        corr = numeric.corr().replace([np.inf, -np.inf], np.nan)

        for i, col1 in enumerate(corr.columns):
            for col2 in corr.columns[i + 1:]:
                value = corr.loc[col1, col2]
                if pd.notna(value):
                    report["correlations"][f"{col1} vs {col2}"] = round(
                        float(value), 4
                    )

    for col in df.columns:
        if df[col].isna().any():
            report["warnings"].append(
                f"{col} contains missing values."
            )

    if report["dataset"]["duplicate_rows"]:
        report["warnings"].append(
            "Duplicate rows were detected; they were not automatically removed."
        )

    return df, report


def report_to_text(report):
    return json.dumps(report, indent=2, ensure_ascii=False)


In [3]:

from google.colab import files

uploaded = files.upload()

filename = next(iter(uploaded))
df = load_dataset(filename)

df, dataset_report = profile_dataset(df)

print("Dataset preview:")
display(df.head())

print("\nDataset profile:")
print(report_to_text(dataset_report)[:15000])


Saving customer_churn_dataset-testing-master.csv to customer_churn_dataset-testing-master.csv
Dataset preview:


,CustomerID,Age,Gender,Tenure,Usage Frequency,Support Calls,Payment Delay,Subscription Type,Contract Length,Total Spend,Last Interaction,Churn
0,1,22,Female,25,14,4,27,Basic,Monthly,598,9,1
1,2,41,Female,28,28,7,13,Standard,Monthly,584,20,0
2,3,47,Male,27,10,2,29,Premium,Annual,757,21,0
3,4,35,Male,9,12,5,17,Premium,Quarterly,232,18,0
4,5,53,Female,58,24,9,2,Standard,Annual,533,18,0



Dataset profile:
{
  "dataset": {
    "rows": 64374,
    "columns": 12,
    "duplicate_rows": 0,
    "memory_mb": 7.06
  },
  "columns": {
    "CustomerID": {
      "dtype": "int64",
      "missing_count": 0,
      "missing_percent": 0.0,
      "unique_values": 64374,
      "statistics": {
        "mean": 32187.5,
        "median": 32187.5,
        "std": 18583.317451413244,
        "min": 1.0,
        "max": 64374.0,
        "q25": 16094.25,
        "q75": 48280.75
      }
    },
    "Age": {
      "dtype": "int64",
      "missing_count": 0,
      "missing_percent": 0.0,
      "unique_values": 48,
      "statistics": {
        "mean": 41.97098207350794,
        "median": 42.0,
        "std": 13.924911309026221,
        "min": 18.0,
        "max": 65.0,
        "q25": 30.0,
        "q75": 54.0
      }
    },
    "Gender": {
      "dtype": "str",
      "missing_count": 0,
      "missing_percent": 0.0,
      "unique_values": 2,
      "top_values": {
        "Female": 34353,
        "Mal

In [4]:

from pydantic import BaseModel, Field
from typing import List, Optional
from langchain_core.output_parsers import PydanticOutputParser


class Hypothesis(BaseModel):
    title: str = Field(description="Short hypothesis title")
    question: str = Field(description="Question to investigate")
    null_hypothesis: str = Field(description="The statistical null hypothesis")
    alternative_hypothesis: str = Field(
        description="The statistical alternative hypothesis"
    )
    variables: List[str] = Field(
        description="Exact column names from the dataset"
    )
    suggested_test: str = Field(
        description=(
            "One of: mann_whitney_u, t_test, chi_square, "
            "spearman, pearson, kruskal_wallis, descriptive_only"
        )
    )
    rationale: str = Field(description="Why this hypothesis is useful")
    limitations: str = Field(
        description="Important assumptions or limitations"
    )


class AnalysisPlan(BaseModel):
    dataset_summary: str
    data_quality_findings: List[str]
    recommended_questions: List[str]
    hypotheses: List[Hypothesis]


parser = PydanticOutputParser(pydantic_object=AnalysisPlan)

print(parser.get_format_instructions())


The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"$defs": {"Hypothesis": {"properties": {"title": {"description": "Short hypothesis title", "title": "Title", "type": "string"}, "question": {"description": "Question to investigate", "title": "Question", "type": "string"}, "null_hypothesis": {"description": "The statistical null hypothesis", "title": "Null Hypothesis", "type": "string"}, "alternative_hypothesis": {"description": "The statistical alternative hypothesis", "title": "Alternative Hypothesis", "type": "string"}, "variables": {"description": "Exact column names from the dataset", "i

In [5]:

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

analysis_prompt = PromptTemplate(
    template="""
You are the analysis planner for Intellianalyzer.

Analyze the dataset profile below and produce a useful, realistic
exploratory analysis plan.

Rules:
- Use only column names that exist in the profile.
- Do not invent sample sizes, p-values, test statistics, or results.
- Suggest hypotheses that can reasonably be investigated.
- Choose descriptive_only when no appropriate test is apparent.
- Select no more than 5 hypotheses.
- For numerical variables, consider their distribution and groups.
- For categorical variables, consider association tests.
- Distinguish correlation from causation.
- Mention relevant assumptions and limitations.
- Return valid structured output matching the schema.

DATASET PROFILE:
{profile}

{format_instructions}
""",
    input_variables=["profile"],
    partial_variables={
        "format_instructions": parser.get_format_instructions()
    }
)


def generate_analysis_plan(report):
    prompt = analysis_prompt.format(
        profile=report_to_text(report)
    )

    raw_output = ask_qwen(prompt, max_new_tokens=1400)

    # Validate the generated JSON using Pydantic.
    # If validation fails, show the raw response for debugging.
    try:
        return parser.parse(raw_output)
    except Exception as exc:
        print("Could not parse Qwen's response:", exc)
        print("Raw model response:\n", raw_output)
        raise


analysis_plan = generate_analysis_plan(dataset_report)

print(analysis_plan.model_dump_json(indent=2))


{
  "dataset_summary": "A dataset of 64,374 customer records with 12 columns, including demographic, behavioral, and service-related features. All variables are complete with no missing data. The dataset includes numerical and categorical variables with clear distributions and a binary churn indicator.",
  "data_quality_findings": [
    "No duplicate rows detected, indicating data integrity.",
    "All variables have 0% missing values, ensuring completeness.",
    "CustomerID is uniquely assigned and spans the full range from 1 to 64,374, suggesting a one-to-one mapping with records.",
    "Churn is binary (0 or 1), with a median of 0 and mean of ~0.47, indicating a significant churn rate.",
    "Subscription Type and Contract Length are categorical with balanced distribution across levels."
  ],
  "recommended_questions": [
    "Is there a relationship between support calls and churn?",
    "Does subscription type influence customer churn?",
    "How does tenure relate to total spend 

In [6]:

from scipy import stats
from scipy.stats import chi2_contingency
from itertools import combinations


def run_hypothesis_test(df, hypothesis):
    h = hypothesis.model_dump() if hasattr(
        hypothesis, "model_dump"
    ) else hypothesis

    columns = h["variables"]
    test_name = h["suggested_test"]

    if any(c not in df.columns for c in columns):
        return {"status": "rejected", "reason": "Unknown column name"}

    if test_name == "descriptive_only":
        return {
            "status": "not_tested",
            "reason": "This hypothesis requires descriptive analysis."
        }

    try:
        if test_name in ("pearson", "spearman"):
            if len(columns) != 2:
                raise ValueError("Correlation requires exactly 2 columns.")

            x = pd.to_numeric(df[columns[0]], errors="coerce")
            y = pd.to_numeric(df[columns[1]], errors="coerce")
            valid = pd.DataFrame({"x": x, "y": y}).replace(
                [np.inf, -np.inf], np.nan
            ).dropna()

            if len(valid) < 3:
                raise ValueError("Not enough valid observations.")
            if valid["x"].nunique() < 2 or valid["y"].nunique() < 2:
                raise ValueError("A constant variable cannot be correlated.")

            fn = stats.pearsonr if test_name == "pearson" else stats.spearmanr
            result = fn(valid["x"], valid["y"])
            statistic, p_value = float(result.statistic), float(result.pvalue)
            n = len(valid)

        elif test_name in ("mann_whitney_u", "t_test"):
            if len(columns) != 2:
                raise ValueError(
                    "This test requires one numeric and one grouping column."
                )

            a, b = columns
            numeric_col = a if pd.api.types.is_numeric_dtype(df[a]) else b
            group_col = b if numeric_col == a else a

            if not pd.api.types.is_numeric_dtype(df[numeric_col]):
                raise ValueError("A numeric variable is required.")

            clean = df[[numeric_col, group_col]].dropna()
            groups = clean[group_col].unique()

            if len(groups) != 2:
                raise ValueError("Exactly two groups are required.")

            x = pd.to_numeric(
                clean.loc[clean[group_col] == groups[0], numeric_col],
                errors="coerce"
            ).dropna()
            y = pd.to_numeric(
                clean.loc[clean[group_col] == groups[1], numeric_col],
                errors="coerce"
            ).dropna()

            if min(len(x), len(y)) < 2:
                raise ValueError("Each group needs at least 2 valid observations.")

            if test_name == "mann_whitney_u":
                result = stats.mannwhitneyu(
                    x, y, alternative="two-sided"
                )
            else:
                result = stats.ttest_ind(
                    x, y, equal_var=False, nan_policy="omit"
                )

            statistic, p_value = float(result.statistic), float(result.pvalue)
            n = len(x) + len(y)

        elif test_name == "chi_square":
            if len(columns) != 2:
                raise ValueError("Chi-square requires exactly 2 columns.")

            table = pd.crosstab(df[columns[0]], df[columns[1]])
            if table.shape[0] < 2 or table.shape[1] < 2:
                raise ValueError("Both columns must have at least 2 categories.")

            statistic, p_value, dof, expected = chi2_contingency(table)
            statistic, p_value = float(statistic), float(p_value)
            n = int(table.to_numpy().sum())

        elif test_name == "kruskal_wallis":
            if len(columns) != 2:
                raise ValueError(
                    "Kruskal-Wallis requires a numeric variable and a group."
                )

            a, b = columns
            numeric_col = a if pd.api.types.is_numeric_dtype(df[a]) else b
            group_col = b if numeric_col == a else a

            if not pd.api.types.is_numeric_dtype(df[numeric_col]):
                raise ValueError("A numeric variable is required.")

            clean = df[[numeric_col, group_col]].dropna()
            groups = [
                pd.to_numeric(
                    clean.loc[clean[group_col] == g, numeric_col],
                    errors="coerce"
                ).dropna().to_numpy()
                for g in clean[group_col].unique()
            ]
            groups = [g for g in groups if len(g) > 0]

            if len(groups) < 2:
                raise ValueError("At least 2 non-empty groups are required.")

            result = stats.kruskal(*groups)
            statistic, p_value = float(result.statistic), float(result.pvalue)
            n = sum(map(len, groups))

        else:
            raise ValueError(f"Unsupported test: {test_name}")

        return {
            "status": "success",
            "test": test_name,
            "variables": columns,
            "sample_size": int(n),
            "statistic": statistic,
            "p_value": p_value,
            "significant_at_0_05": bool(p_value < 0.05),
            "interpretation_note": (
                "Statistical significance does not establish causation. "
                "Check test assumptions and consider multiple-testing correction."
            )
        }

    except Exception as exc:
        return {
            "status": "error",
            "test": test_name,
            "variables": columns,
            "reason": str(exc)
        }


test_results = []

for hypothesis in analysis_plan.hypotheses:
    result = run_hypothesis_test(df, hypothesis)
    test_results.append({
        "hypothesis": hypothesis.title,
        "question": hypothesis.question,
        "result": result
    })

for item in test_results:
    print(json.dumps(item, indent=2, default=str))
    print("-" * 60)


{
  "hypothesis": "Support Calls and Churn",
  "question": "Is there a relationship between the number of support calls and customer churn?",
  "result": {
    "status": "success",
    "test": "pearson",
    "variables": [
      "Support Calls",
      "Churn"
    ],
    "sample_size": 64374,
    "statistic": 0.3046311367131596,
    "p_value": 0.0,
    "significant_at_0_05": true,
    "interpretation_note": "Statistical significance does not establish causation. Check test assumptions and consider multiple-testing correction."
  }
}
------------------------------------------------------------
{
  "hypothesis": "Subscription Type and Churn",
  "question": "Does subscription type affect the likelihood of churn?",
  "result": {
    "status": "success",
    "test": "chi_square",
    "variables": [
      "Subscription Type",
      "Churn"
    ],
    "sample_size": 64374,
    "statistic": 13.655943999628535,
    "p_value": 0.0010830523222382482,
    "significant_at_0_05": true,
    "interpret

In [7]:

from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)


def build_knowledge_base(report, plan, results):
    documents = []

    documents.append(Document(
        page_content="DATASET PROFILE\n" + report_to_text(report),
        metadata={"source": "dataset_profile"}
    ))

    for h in plan.hypotheses:
        documents.append(Document(
            page_content=(
                "HYPOTHESIS\n"
                + h.model_dump_json(indent=2)
            ),
            metadata={"source": "hypothesis", "title": h.title}
        ))

    for result in results:
        documents.append(Document(
            page_content=(
                "STATISTICAL TEST RESULT\n"
                + json.dumps(result, indent=2, default=str)
            ),
            metadata={
                "source": "statistical_result",
                "title": result["hypothesis"]
            }
        ))

    return FAISS.from_documents(documents, embedding_model)


vector_store = build_knowledge_base(
    dataset_report, analysis_plan, test_results
)

print("Knowledge base created successfully.")


/tmp/ipykernel_3046/735508542.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Knowledge base created successfully.


In [8]:

def ask_dataset(question, k=4):
    retrieved_docs = vector_store.similarity_search(
        question, k=k
    )

    context = "\n\n".join(
        f"[Source: {doc.metadata.get('source', 'analysis')}]\n"
        f"{doc.page_content}"
        for doc in retrieved_docs
    )

    prompt = f"""
You are Intellianalyzer's dataset question-answering assistant.

Answer the user's question using the retrieved analysis below.

Rules:
- Use only evidence supported by the context.
- Never invent statistics or claim an untested hypothesis was verified.
- Distinguish statistical association from causation.
- If evidence is missing, say what information is missing.
- Cite the relevant source type in your answer.
- Be concise and helpful.

RETRIEVED ANALYSIS:
{context}

USER QUESTION:
{question}

ANSWER:
"""

    answer = ask_qwen(prompt, max_new_tokens=450)

    return {
        "answer": answer,
        "sources": [
            {
                "source": doc.metadata.get("source"),
                "title": doc.metadata.get("title", "")
            }
            for doc in retrieved_docs
        ]
    }


response = ask_dataset(
    "What are the most important findings in this dataset?"
)

print(response["answer"])
print("\nRetrieved sources:")
print(json.dumps(response["sources"], indent=2))


The most important findings in this dataset are:

1. **Gender and Churn**: There is a statistically significant association between gender and churn (p-value not explicitly provided, but hypothesis testing is suggested). However, the dataset does not provide a breakdown of churn rates by gender, so a direct comparison cannot be made from the available statistics.

2. **Usage Frequency and Churn**: Higher usage frequency is significantly associated with lower churn (Spearman correlation: -0.117, p < 0.001). This indicates that more frequent usage is linked to reduced churn risk, suggesting customer engagement may be a key factor in retention.

3. **Payment Delay and Churn**: There is a strong positive correlation between payment delay and churn (correlation: 0.557), indicating that customers who delay payments are significantly more likely to churn.

4. **Churn Probability**: The overall churn rate is 47.4% (mean of Churn column), with a median of 0.0, suggesting a high risk of churn in

In [9]:

%pip install -q fastapi uvicorn nest-asyncio


In [19]:

import re
import numpy as np
import pandas as pd
from pydantic import BaseModel
from fastapi import HTTPException


class ChatRequest(BaseModel):
    question: str


def calculate_direct_answer(question, df):
    q = question.lower().strip()

    # Match phrases such as:
    # "above age 28"
    # "above the age of 28"
    # "older than 28"
    # "greater than 28"
    # "more than 28"
    threshold_match = re.search(
        r"\b(?:above|over|older than|greater than|more than)\b"
        r"(?:\s+the)?(?:\s+age)?(?:\s+of)?\s+"
        r"(-?\d+(?:\.\d+)?)",
        q
    )

    # Find the column explicitly mentioned, if any.
    column = next(
        (
            col for col in df.columns
            if re.search(
                rf"(?<!\w){re.escape(str(col).lower())}(?!\w)",
                q
            )
        ),
        None
    )

    # Recognize age-related questions even when "Age" is not
    # explicitly written as a column name.
    if column is None and re.search(
        r"\b(age|older|younger|members|customers)\b", q
    ):
        column = next(
            (
                col for col in df.columns
                if str(col).strip().lower() in {
                    "age", "customer age", "member age"
                }
            ),
            None
        )

    # Recognize "how many customers/members/rows..."
    asks_count = bool(re.search(
        r"\b(how many|count|number of|total number)\b", q
    ))

    if column is None:
        return None

    series = pd.to_numeric(df[column], errors="coerce")
    series = series.replace([np.inf, -np.inf], np.nan)

    # Exact count above a threshold.
    if asks_count and threshold_match:
        threshold = float(threshold_match.group(1))
        count = int((series > threshold).sum())

        return {
            "type": "computed_result",
            "column": str(column),
            "operation": "count_greater_than",
            "threshold": threshold,
            "count": count,
            "valid_values": int(series.notna().sum()),
            "answer": (
                f"There are {count:,} records with {column} > "
                f"{threshold:g}, calculated directly from the dataset."
            )
        }

    # Exact count below a threshold.
    below_match = re.search(
        r"\b(?:below|under|younger than|less than)\b"
        r"(?:\s+the)?(?:\s+age)?(?:\s+of)?\s+"
        r"(-?\d+(?:\.\d+)?)",
        q
    )

    if asks_count and below_match:
        threshold = float(below_match.group(1))
        count = int((series < threshold).sum())

        return {
            "type": "computed_result",
            "column": str(column),
            "operation": "count_less_than",
            "threshold": threshold,
            "count": count,
            "valid_values": int(series.notna().sum()),
            "answer": (
                f"There are {count:,} records with {column} < "
                f"{threshold:g}, calculated directly from the dataset."
            )
        }

    # Exact count equal to a value.
    equal_match = re.search(
        r"\b(?:equal to|exactly|is|equals?)\s+(-?\d+(?:\.\d+)?)\b",
        q
    )

    if asks_count and equal_match:
        value = float(equal_match.group(1))
        count = int((series == value).sum())

        return {
            "type": "computed_result",
            "column": str(column),
            "operation": "count_equal_to",
            "value": value,
            "count": count,
            "answer": f"There are {count:,} records where {column} = {value:g}."
        }

    return None


@api.post("/chat")
def chat(request: ChatRequest):
    if not app_state["ready"]:
        raise HTTPException(
            status_code=400,
            detail="Analyze a dataset before asking questions."
        )

    question = request.question.strip()

    if not question or len(question) > 2000:
        raise HTTPException(
            status_code=400,
            detail="Enter a question of at most 2000 characters."
        )

    try:
        df = app_state["df"]

        # Calculate directly instead of relying on vector retrieval.
        computed = calculate_direct_answer(question, df)

        if computed is not None:
            # Return the verified result directly. Qwen cannot
            # accidentally replace or contradict the numerical answer.
            return {
                "answer": computed["answer"],
                "computed_result": computed,
                "sources": [{
                    "source": "live_dataframe_computation",
                    "title": "Exact calculation on uploaded dataset"
                }]
            }

        # For explanatory questions, use RAG + Qwen.
        documents = app_state["vector_store"].similarity_search(
            question, k=5
        )

        context = "\n\n".join(
            f"[Source: {doc.metadata.get('source', 'analysis')}]\n"
            f"{doc.page_content}"
            for doc in documents
        )

        prompt = f"""
You are Intellianalyzer, a data analysis assistant.

Answer using the retrieved evidence below.
Never invent numerical results.
Distinguish correlation from causation.
If the evidence is insufficient, say so.
Keep the answer clear and concise.

RETRIEVED EVIDENCE:
{context}

USER QUESTION:
{question}
"""

        answer = ask_qwen(prompt, max_new_tokens=400)

        return {
            "answer": answer,
            "computed_result": None,
            "sources": [
                {
                    "source": doc.metadata.get("source"),
                    "title": doc.metadata.get("title", "")
                }
                for doc in documents
            ]
        }

    except Exception as exc:
        raise HTTPException(
            status_code=500,
            detail=f"Chat failed: {type(exc).__name__}: {str(exc)[:300]}"
        )


In [20]:

from getpass import getpass
from pyngrok import ngrok
import nest_asyncio
import requests

nest_asyncio.apply()

NGROK_TOKEN = "3KCPvunMgHlmrcIZUPZ4s3hi0LC_5pUVico6nozVAG9pSXoXb"
ngrok.set_auth_token(NGROK_TOKEN)

# Close previous tunnels if this cell is rerun.
ngrok.kill()

def run_api():
    uvicorn.run(
        api,
        host="0.0.0.0",
        port=8000,
        log_level="warning",
    )

api_thread = threading.Thread(
    target=run_api,
    daemon=True
)
api_thread.start()

# Wait for FastAPI to start.
for attempt in range(30):
    try:
        response = requests.get(
            "http://127.0.0.1:8000/health",
            timeout=2
        )
        if response.ok:
            break
    except requests.RequestException:
        time.sleep(1)
else:
    raise RuntimeError("FastAPI failed to start on port 8000.")

tunnel = ngrok.connect(8000, "http")
API_URL = tunnel.public_url.rstrip("/")

print("API URL:", API_URL)
print("Health check:", f"{API_URL}/health")
print("API documentation:", f"{API_URL}/docs")


ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use


API URL: https://amplify-ivy-poster.ngrok-free.dev
Health check: https://amplify-ivy-poster.ngrok-free.dev/health
API documentation: https://amplify-ivy-poster.ngrok-free.dev/docs
